# Task 4 — NLP Sentiment Classification (TF-IDF)
**Datheon Machine Learning & Data Science Internship**

| Property | Value |
|---|---|
| **Dataset** | `sentiment_reviews.csv` (provided — 10 reviews) |
| **Models** | Multinomial Naive Bayes · Logistic Regression |
| **Vectorizer** | TF-IDF (max_features=500, unigrams+bigrams, sublinear_tf) |
| **CV Strategy** | Leave-One-Out (LOO) — optimal for n<30 |
| **Metrics** | Accuracy · Precision · Recall · F1 (weighted) |


## 1. Import Libraries

In [ ]:
import warnings, os, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    for pkg in ['stopwords','wordnet','punkt','omw-1.4']: nltk.download(pkg, quiet=True)
    STOPWORDS  = set(stopwords.words('english'))
    LEMMATIZER = WordNetLemmatizer()
    print('NLTK available ✔')
except ImportError:
    STOPWORDS  = {'i','me','the','a','an','and','or','is','are','was','were','in','on','at','to','for'}
    LEMMATIZER = None
    print('NLTK not installed — basic cleaning only')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COLORS = {'positive':'#55A868','negative':'#DD8452','neutral':'#4C72B0'}
SEED   = 42

## 2. Load Dataset (provided CSV only)

In [ ]:
# Try same directory as this notebook first, then uploaded path
NB_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
PATHS  = [
    os.path.join(NB_DIR, 'sentiment_reviews.csv'),
    r'C:\Users\albert\.gemini\antigravity\brain\d2b5812c-f7b5-4143-a723-c7f636a38834\.user_uploaded\media_1795360003844_374a95b1.csv',
]
for path in PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path); print(f'Loaded from: {path}'); break

df.columns = [c.strip().lower() for c in df.columns]
if 'review_text' in df.columns: df = df.rename(columns={'review_text': 'text'})
df['sentiment'] = df['sentiment'].str.strip().str.lower()
df = df[df['sentiment'].isin(['positive','negative','neutral'])].reset_index(drop=True)

print(f'\nShape: {df.shape}')
df

## 3. Class Distribution

In [ ]:
counts = df['sentiment'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index], edgecolor='white', width=0.5)
ax.bar_label(bars, padding=3, fontsize=12, fontweight='bold')
ax.set_title('Sentiment Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylim(0, counts.max()*1.3)
plt.tight_layout(); plt.savefig('nlp_1_class_dist.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. Text Cleaning Pipeline
> HTML strip → lowercase → punctuation removal → stopword filtering → lemmatization

In [ ]:
def clean_text(text):
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = re.sub(r'http\S+', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    if LEMMATIZER: tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)

# Display before / after
for _, row in df[['text','clean_text','sentiment']].iterrows():
    print(f"[{row['sentiment']:8s}] Raw  : {row['text'][:70]}")
    print(f"           Clean: {row['clean_text']}\n")

## 5. TF-IDF Vectorization

In [ ]:
le      = LabelEncoder()
y       = le.fit_transform(df['sentiment'])
classes = le.classes_
X_text  = df['clean_text'].values

tfidf   = TfidfVectorizer(max_features=500, ngram_range=(1,2), sublinear_tf=True)
X_tfidf = tfidf.fit_transform(X_text)

print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'TF-IDF matrix: {X_tfidf.shape}')

## 6. Top TF-IDF Terms per Class

In [ ]:
present_classes = [c for c in classes if (df['sentiment']==c).sum()>0]
fig, axes = plt.subplots(1, len(present_classes), figsize=(6*len(present_classes), 5))
if len(present_classes)==1: axes=[axes]
fig.suptitle('Top TF-IDF Terms by Sentiment', fontsize=14, fontweight='bold')
for ax, cls in zip(axes, present_classes):
    idx    = le.transform([cls])[0]
    mask   = y == idx
    scores = np.asarray(X_tfidf[mask].mean(axis=0)).flatten()
    top_n  = min(10, (scores>0).sum())
    if top_n == 0: continue
    ti     = scores.argsort()[-top_n:][::-1]
    terms  = [tfidf.get_feature_names_out()[i] for i in ti]
    vals   = scores[ti]
    ax.barh(terms[::-1], vals[::-1], color=COLORS.get(cls,'#888'), edgecolor='white')
    ax.set_title(f"'{cls}'  (n={mask.sum()})", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('nlp_2_tfidf_terms.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. Model Training — Leave-One-Out Cross-Validation
> LOO is the standard approach for very small datasets (n=10): every sample is held out exactly once.

In [ ]:
PIPE_CONFIGS = {
    'Multinomial NB': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1,2), sublinear_tf=True)),
        ('clf',   MultinomialNB(alpha=0.5)),
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1,2), sublinear_tf=True)),
        ('clf',   LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', multi_class='auto', random_state=SEED)),
    ]),
}

loo     = LeaveOneOut()
results = {}

for name, pipe in PIPE_CONFIGS.items():
    y_pred = cross_val_predict(pipe, X_text, y, cv=loo)
    pipe.fit(X_text, y)
    results[name] = {
        'pipe': pipe, 'y_pred': y_pred,
        'accuracy':  accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y, y_pred, average='weighted', zero_division=0),
    }
    print(f'\n── {name} ──')
    print(classification_report(y, y_pred, target_names=classes, digits=4, zero_division=0))

## 8. Confusion Matrices (LOO predictions)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — LOO Predictions', fontsize=14, fontweight='bold')
for ax, (name, r) in zip(axes, results.items()):
    ConfusionMatrixDisplay(confusion_matrix(y, r['y_pred']), display_labels=classes).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}  Acc={r["accuracy"]:.4f}', fontsize=11)
plt.tight_layout(); plt.savefig('nlp_3_confusion.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Benchmark Bar Chart

In [ ]:
metrics    = ['accuracy','precision','recall','f1']
met_labels = ['Accuracy','Precision','Recall','F1-Score']
bench_data = {n: [results[n][m] for m in metrics] for n in results}
bench_df   = pd.DataFrame(bench_data, index=met_labels)
fig, ax    = plt.subplots(figsize=(10, 5))
bench_df.T.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#55A868','#c0785c'], edgecolor='white', width=0.65)
ax.set_title('NLP Benchmark — LOO-CV (n=10)', fontsize=13, fontweight='bold')
ax.set_xticklabels(list(results.keys()), rotation=0); ax.set_ylim(0, 1.15); ax.legend(title='Metric')
for c in ax.containers: ax.bar_label(c, fmt='%.3f', fontsize=8, padding=2)
plt.tight_layout(); plt.savefig('nlp_4_benchmark.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. Interactive: predict_sentiment()

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best_pipe = results[best_name]['pipe']
print(f'Best model: {best_name}  (LOO Accuracy={results[best_name]["accuracy"]:.4f})')

def predict_sentiment(text):
    cleaned  = clean_text(text)
    pred_idx = best_pipe.predict([cleaned])[0]
    proba    = best_pipe.predict_proba([cleaned])[0]
    label    = le.inverse_transform([pred_idx])[0]
    conf     = proba[pred_idx]
    return {'sentiment': label, 'confidence': round(float(conf),4),
            'probabilities': {le.classes_[i]: round(float(p),4) for i,p in enumerate(proba)}}

# Test on provided reviews
print('\nPredictions on provided reviews:')
for _, row in df.iterrows():
    r   = predict_sentiment(row['text'])
    chk = '✓' if r['sentiment']==row['sentiment'] else '✗'
    print(f"{chk} True:[{row['sentiment']:8s}] Pred:[{r['sentiment']:8s}] Conf:{r['confidence']:.2%}")

In [ ]:
# Try your own phrases:
test_phrases = [
    'The product is absolutely amazing and works perfectly!',
    'Terrible quality, broken on arrival, complete waste of money.',
    'It is okay. Nothing special but does the basic job fine.',
]
for phrase in test_phrases:
    r = predict_sentiment(phrase)
    print(f"Input  : {phrase}")
    print(f"Result : [{r['sentiment'].upper():10s}]  Conf: {r['confidence']:.2%}")
    print(f"Probs  : {r['probabilities']}\n")

## 11. Save Model to Pickle File
> Export trained NLP pipeline and label encoder for deployment.


In [ ]:
import pickle, os
os.makedirs('../models', exist_ok=True)
nlp_bundle = {
    'pipeline': best_pipe,
    'label_encoder': le,
    'classes': list(classes)
}
with open('../models/sentiment_model.pkl', 'wb') as f:
    pickle.dump(nlp_bundle, f)
print('✔ Sentiment model exported to ../models/sentiment_model.pkl')
